# Daily session features
Productionized as a scheduled papermill job:
`papermill sessions_daily.ipynb /tmp/out.ipynb -p as_of 2024-03-01`

In [1]:
as_of = "2024-03-01"
lookback_days = 90
events_path = "s3://analytics-prod/events/"
output_path = "s3://analytics-prod/features/sessions/"

In [2]:
import pandas as pd
import numpy as np

events = pd.read_parquet(events_path)
events["ts"] = pd.to_datetime(events["ts"], utc=True)

In [3]:
as_of_ts = pd.Timestamp(as_of, tz="UTC")
window_start = as_of_ts - pd.Timedelta(days=lookback_days)
events = events[(events["ts"] >= window_start) & (events["ts"] < as_of_ts)]
events = events.sort_values(["user_id", "ts"])

In [8]:
new_session = events.groupby("user_id")["ts"].diff() > pd.Timedelta(minutes=30)
events["session_id"] = (
    new_session.groupby(events["user_id"]).cumsum().fillna(0).astype(int)
)
sessionized = (
    events.groupby(["user_id", "session_id"])
    .agg(events_in_session=("ts", "size"), session_start=("ts", "min"))
    .reset_index()
)
user_sessions = (
    sessionized.groupby("user_id")
    .agg(sessions=("session_id", "nunique"), total_events=("events_in_session", "sum"))
    .reset_index()
)

In [5]:
events = events.drop_duplicates(subset=["user_id", "ts", "event_name"])
events["ts"] = events["ts"].dt.tz_convert("UTC").dt.tz_localize(None)

In [9]:
user_sessions["as_of"] = as_of
user_sessions.to_parquet(
    output_path + f"as_of={as_of}/features.parquet", index=False
)